# LSTM Sanity Checks (CLASSIFIER)

Runs four ablations on the *same* held-out test set against an already-trained GELSTM checkpoint, and tabulates them side-by-side:

1. **Baseline** — original config (`use_time_delta=True`, full visit history, true temporal order).
2. **Shuffled visit order** — visits per subject randomly permuted at eval. Δt rides with its original visit so the Δt distribution is preserved while order is destroyed.
3. **Δt removed** — `use_time_delta=False` at eval.
4. **Fixed sequence length (first-2 visits)** — uses `LongitudinalSubjectDataset(max_visits=2, require_full_window=True)`.
5. **Metadata-only baseline** — copied across from `SANITY_TIME_METADATA_BASELINE.ipynb` for cross-reference.

*All four LSTM rows use the same trained checkpoint to isolate the effect of each ablation.*

In [ ]:
# === Papermill parameters (injected by run_experiment.py) ===
# Safe interactive defaults: None keeps the original Jupyter behaviour
# (interactive checkpoint/threshold prompts, JSON-config loading).
EXPERIMENT_ID = None
MODE = None
MODEL = None
DATASET = None
SEED = 42
GAAE_CHECKPOINT_PATH = None   # None -> interactive checkpoint picker
THRESHOLD_MODE = None         # None -> interactive prompt; else youden | best-f1 | fixed
FIXED_THRESHOLD = None        # required when THRESHOLD_MODE is fixed
WANDB_ENABLED = True          # W&B logging is on by default
OUTPUT_DIR = None             # defaults to outputs/<experiment-id>/ when run standalone
RESOLVED_CONFIG = None        # merged hyperparameter dict; overrides on-disk JSON when set
RUN_DIR = None                # set by the runner: where run_summary.json / artifacts go
RUN_NAME = None               # set by the runner: the W&B run name


In [ ]:

import sys
import importlib
from pathlib import Path
import json, os, copy
import numpy as np
import pandas as pd
import torch

V2_ROOT = Path('/mnt/e/fyassine/ad-early-detection/CLASSIFIER')
if str(V2_ROOT) not in sys.path:
    sys.path.insert(0, str(V2_ROOT))

# Force reload so edits to model source are picked up without restarting the kernel.
import model.GELSTM.utils
import model.GELSTM.train
importlib.reload(model.GELSTM.utils)
importlib.reload(model.GELSTM.train)

from model.GELSTM.models  import GELSTMClassifier
from model.GELSTM.dataset import LongitudinalSubjectDataset
from model.GELSTM.train   import evaluate, make_batches
from common.sanity        import run_full_audit

RANDOM_STATE = 42
set_seed(RANDOM_STATE)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


In [ ]:
import sys
if '/mnt/e/fyassine/ad-early-detection' not in sys.path:
    sys.path.insert(0, '/mnt/e/fyassine/ad-early-detection')
from DATA.src.splitting.load_splits import splits_dir, split_csv_paths


import json
from pathlib import Path

from common.sanity import run_full_audit

DATA_VERSION = '__fc_wholebrain_sch200_flat__'
DATA_ROOT    = Path('/mnt/e/fyassine/ad-early-detection/DATA/DELCODE') / DATA_VERSION
WB_DATA_ROOT = str(DATA_ROOT / 'matrices')
METADATA_DIR = DATA_ROOT / 'metadata'
COHORTS_CSV  = str(METADATA_DIR / 'cohorts.csv')

SPLIT_CSVS = {
    'train': str(splits_dir('downstream') / 'train.csv'),
    'val':   str(splits_dir('downstream') / 'val.csv'),
    'test':  str(splits_dir('downstream') / 'test.csv'),
}
_ = run_full_audit(SPLIT_CSVS)

GAAE_CONFIG_PATH = Path('/mnt/e/fyassine/ad-early-detection/CLASSIFIER/configs/gaae_delcode_whole_brain.json')

# ── Checkpoint selector ──────────────────────────────────────────────────────
CKPT_ROOT = Path('/mnt/e/fyassine/ad-early-detection/CLASSIFIER/notebooks/checkpoints')
CHECKPOINT_SEARCH_DIRS = sorted(CKPT_ROOT.glob('checkpoints_gelstm*'))

checkpoint_candidates = sorted(
    [(run_dir.name, str(run_dir / f'model_{run_dir.name}.pth'), str(run_dir))
     for ckpt_dir in CHECKPOINT_SEARCH_DIRS
     for run_dir in sorted(Path(ckpt_dir).iterdir()) if run_dir.is_dir()
     if (run_dir / f'model_{run_dir.name}.pth').exists()],
    key=lambda x: x[0],
)
if not checkpoint_candidates:
    raise FileNotFoundError(f'No GELSTM checkpoints found under {CKPT_ROOT}')

print('Available GELSTM checkpoints:')
for i, (name, _, _) in enumerate(checkpoint_candidates):
    print(f'  {i}: {name}')

if GAAE_CHECKPOINT_PATH is not None:
    _t = str(Path(GAAE_CHECKPOINT_PATH).resolve())
    _m = [c for c in checkpoint_candidates if str(Path(c[1]).resolve()) == _t]
    if not _m:
        raise FileNotFoundError(f'GAAE_CHECKPOINT_PATH={GAAE_CHECKPOINT_PATH!r} not among candidates')
    GELSTM_RUN_NAME, GELSTM_CKPT_PATH, GELSTM_RUN_DIR = _m[0]
elif RUN_DIR is not None:
    GELSTM_RUN_NAME, GELSTM_CKPT_PATH, GELSTM_RUN_DIR = checkpoint_candidates[-1]
    print(f'Headless: using latest GELSTM checkpoint {GELSTM_RUN_NAME}')
else:
    selected_idx = int(input('Select checkpoint index: '))
    GELSTM_RUN_NAME, GELSTM_CKPT_PATH, GELSTM_RUN_DIR = checkpoint_candidates[selected_idx]
print(f'Selected: {GELSTM_RUN_NAME}')
print(f'Path:     {GELSTM_CKPT_PATH}')


## Load test set + checkpoint

We build two test datasets: one with **all visits** (used by checks 1-3), one with **first-2-visits and require_full_window=True** (used by check 4).

In [ ]:
test_df = pd.read_csv(SPLIT_CSVS['test'])
test_mci = test_df.copy()
print(f'Test subjects (MCI/converter): {len(test_mci)}')

with open(GAAE_CONFIG_PATH) as f:
    hp = json.load(f)
ADJ_K        = hp.get('adjacency_k', 8)
FILE_VARIANT = hp.get('file_variant', 'z_transformed')

test_ds_all = LongitudinalSubjectDataset(
    WB_DATA_ROOT, test_mci, COHORTS_CSV,
    adjacency_k=ADJ_K, file_variant=FILE_VARIANT,
)
test_ds_first2 = LongitudinalSubjectDataset(
    WB_DATA_ROOT, test_mci, COHORTS_CSV,
    adjacency_k=ADJ_K, file_variant=FILE_VARIANT,
    max_visits=2, require_full_window=True,
)
test_items_all    = [test_ds_all[i]    for i in range(len(test_ds_all))]
test_items_first2 = [test_ds_first2[i] for i in range(len(test_ds_first2))]

In [ ]:

def load_gelstm():
    m = GELSTMClassifier(
        in_features=200, gaae_hidden=hp.get('hidden_dim', 128),
        gaae_latent=hp.get('latent_dim', 64),
        gaae_heads=hp.get('num_heads', 2),
        gaae_cond_dim=hp.get('cond_dim', 2),
        gaae_dropout=hp.get('dropout', 0.3),
        lstm_hidden=128, lstm_layers=2, lstm_dropout=0.3,
        use_time_delta=True, classifier_hidden=64,
    ).to(device)
    state = torch.load(GELSTM_CKPT_PATH, map_location=device)
    if isinstance(state, dict) and 'state_dict' in state:
        state = state['state_dict']
    m.load_state_dict(state, strict=False)
    m.eval()
    print(f'Loaded: {GELSTM_CKPT_PATH}')
    return m


In [ ]:

# ► Select a checkpoint from the dropdown in cell 2, then run this cell.
model = load_gelstm()


In [ ]:

BATCH_SIZE = 16
rng = np.random.default_rng(RANDOM_STATE)
rows = []

def _run(items, **eval_kwargs):
    batches = make_batches(items, BATCH_SIZE, shuffle=False)
    return evaluate(model, batches, device, **eval_kwargs)

# 1. Baseline.
r = _run(test_items_all, use_time_delta=True, shuffle_order=False)
rows.append({'check': '1_baseline', 'auc': r['auc'], 'sens': r['sensitivity'], 'spec': r['specificity']})

# 2. Shuffled visit order (repeat 5x to estimate variance).
aucs = []
for k in range(5):
    r = _run(test_items_all, use_time_delta=True, shuffle_order=True,
             shuffle_rng=np.random.default_rng(RANDOM_STATE + k))
    aucs.append(r['auc'])
rows.append({'check': '2_shuffled_order', 'auc': float(np.mean(aucs)), 'sens': float('nan'),
             'spec': float('nan'), 'auc_std': float(np.std(aucs))})

# 3. Δt zeroed-out (keeps input shape intact; only the signal is removed).
r = _run(test_items_all, use_time_delta=True, zero_time_delta=True, shuffle_order=False)
rows.append({'check': '3_no_delta_t', 'auc': r['auc'], 'sens': r['sensitivity'], 'spec': r['specificity']})

# 4. Fixed sequence length (first 2 visits, full-window).
r = _run(test_items_first2, use_time_delta=True, shuffle_order=False)
rows.append({'check': '4_first_2_visits', 'auc': r['auc'], 'sens': r['sensitivity'],
             'spec': r['specificity'], 'n_subjects': len(test_items_first2)})

pd.DataFrame(rows)


## Cross-reference: metadata-only baseline

Reuse the result table from `SANITY_TIME_METADATA_BASELINE.ipynb` to anchor row 5 of the sanity table — set the value manually after running that notebook.

In [ ]:

METADATA_BASELINE_JSON = Path('/mnt/e/fyassine/ad-early-detection/CLASSIFIER/notebooks/metadata_baseline_results.json')

if METADATA_BASELINE_JSON.exists():
    with open(METADATA_BASELINE_JSON) as f:
        _meta = json.load(f)
    META_ONLY_CV_AUC   = _meta['cv_auc_mean']
    META_ONLY_TEST_AUC = _meta['test_auc']
    print(f'Loaded metadata baseline from {METADATA_BASELINE_JSON}')
    print(f"  best model : {_meta['best_model']}")
    print(f"  cv_auc     : {META_ONLY_CV_AUC:.4f} ± {_meta['cv_auc_std']:.4f}")
    print(f"  test_auc   : {META_ONLY_TEST_AUC:.4f}")
else:
    META_ONLY_CV_AUC   = float('nan')
    META_ONLY_TEST_AUC = float('nan')
    print(f'WARNING: {METADATA_BASELINE_JSON} not found.')
    print('Run SANITY_TIME_METADATA_BASELINE.ipynb first to generate it.')

rows.append({
    'check':   '5_metadata_only_baseline',
    'auc':     META_ONLY_TEST_AUC,
    'cv_auc':  META_ONLY_CV_AUC,
})
summary = pd.DataFrame(rows)

# Display with — in place of NaN so the table is easier to read.
summary.fillna('—').style.set_caption('GELSTM Sanity Checks')


## Reading the table

* **Baseline vs shuffled-order.** If row 2 ≈ row 1, the LSTM is *not* exploiting temporal order — the per-visit graph embedding is doing the work, the recurrence is decorative.
* **Baseline vs Δt-removed.** If row 3 collapses, Δt is carrying real signal (which may itself be leakage; see metadata baseline).
* **Baseline vs first-2-visits.** If row 4 stays high while only seeing the *earliest* two scans, the model is doing *true early detection*. If it collapses, prior numbers were aided by later visits or by sequence-length cues.
* **Metadata-only.** Anchor for everything: subtract this from every fMRI-based row to estimate the marginal contribution of brain features.